# 1. Setup & Clone Repository
Clone the GitHub repository and install dependencies.

In [ ]:
!git clone https://github.com/10Unknownboy/Malaria-Parasite-Detector.git
%cd Malaria-Parasite-Detector
!pip install -r requirements_colab.txt

# 2. Download Dataset via Kagglehub
Download the dataset without an API key and move it to the data directory.

In [ ]:
import kagglehub
import os
import shutil

# Download latest version
path = kagglehub.dataset_download("iarunava/cell-images-for-detecting-malaria")
print("Path to dataset files:", path)

# The downloaded path will contain a 'cell_images' folder, which contains 'Parasitized' and 'Uninfected'
source_dir = os.path.join(path, "cell_images")
if not os.path.exists(os.path.join(source_dir, "Parasitized")):
    # Sometimes kagglehub extracts it differently, check subfolders
    for root, dirs, files in os.walk(path):
        if "Parasitized" in dirs and "Uninfected" in dirs:
            source_dir = root
            break

# Create data directory if it doesn't exist
os.makedirs("data", exist_ok=True)

# Copy the folders
print("Copying Parasitized images...")
!cp -r "$source_dir/Parasitized" data/
print("Copying Uninfected images...")
!cp -r "$source_dir/Uninfected" data/
print("Dataset ready at data/")

# 3. Execute Training Pipeline
Run the local training orchestrator to train all models, evaluate them, and generate Grad-CAMs.

In [ ]:
!python main.py

# 4. Package and Download Results
Move all charts, graphs, and metrics into the `models/` folder, zip it, and download.

In [ ]:
import os
from google.colab import files

# Move results and reports into the models folder so they get zipped together
if os.path.exists("results"):
    !cp -r results models/
if os.path.exists("reports"):
    !cp -r reports models/

print("Compressing models directory...")
!zip -r /content/malaria_models_and_results.zip models/

print("Downloading zip file...")
files.download('/content/malaria_models_and_results.zip')

# 5. Run Streamlit Dashboard
Use Ngrok to expose the Streamlit app to the public internet.

In [ ]:
import getpass
from pyngrok import ngrok, conf
import time

print("Enter your ngrok authtoken (get it for free at https://dashboard.ngrok.com/get-started/your-authtoken):")
authtoken = getpass.getpass()
conf.get_default().auth_token = authtoken

# Terminate open tunnels if they exist
ngrok.kill()

# Open a tunnel on the default Streamlit port (8501)
public_url = ngrok.connect(8501).public_url
print(f"\n✅ Streamlit Dashboard is starting! It will be live at: {public_url}\n")

# Run Streamlit in the background
!nohup streamlit run app.py > streamlit_logs.txt 2>&1 &
time.sleep(3)
print("Dashboard is running. Check the URL above!")